# calculate&angle_change.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import math
from decimal import Decimal
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from google.colab import files
import io


In [ ]:
uploaded = files.upload()

all_data = pd.DataFrame()


In [ ]:
for file_name in uploaded.keys():
    if file_name.endswith(".csv"):

        df = pd.read_csv(io.BytesIO(uploaded[file_name]))

        all_data = pd.concat([all_data, df], ignore_index=True)


In [ ]:
kpts_columns = all_data['kpts'].str.split(',', expand=True)

for i in range(kpts_columns.shape[1]):
    kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


In [ ]:
all_data = all_data.join(kpts_columns, rsuffix='_kpts')


In [ ]:
def angle_change(x1, y1, x2, y2, x3, y3):

    a_x = x2 - x1
    a_y = y2 - y1

    b_x = x3 - x2
    b_y = y3 - y2

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = math.sqrt(a_x**2 + a_y**2)
    magnitude_b = math.sqrt(b_x**2 + b_y**2)
    if magnitude_a * magnitude_b == 0:
        return 0

    cos_angle = dot_product / (magnitude_a * magnitude_b)

    cos_angle = max(-1, min(1, cos_angle))

    return math.acos(cos_angle) * (180 / math.pi)


In [ ]:
x11 = all_data['kpt_31'].astype(float)
y11 = all_data['kpt_32'].astype(float)
x12 = all_data['kpt_34'].astype(float)
y12 = all_data['kpt_35'].astype(float)

x14 = all_data['kpt_40'].astype(float)
y14 = all_data['kpt_41'].astype(float)
x15 = all_data['kpt_44'].astype(float)
y15 = all_data['kpt_45'].astype(float)
x18 = (x11 + x12) / 2
y18 = (y11 + y12) / 2


In [ ]:
all_data['kpt_52'] = x18
all_data['kpt_53'] = y18


In [ ]:
angle_changes = []


In [ ]:
for i in range(len(x18)):
    angle_change_value = angle_change(x14[i], y14[i], x18[i], y18[i], x15[i], y15[i])
    print(f"x14[{i}] = {x14[i]}, y14[{i}] = {y14[i]}, x18[{i}] = {x18[i]}, y18[{i}] = {y18[i]}, x15[{i}] = {x15[i]}, y15[{i}] = {y15[i]}, angle_change = {angle_change_value}")
    angle_changes.append(angle_change_value)


In [ ]:
angle_changes.insert(0, 0)
angle_changes.append(0)


In [ ]:
if len(angle_changes) > len(all_data):
    angle_changes_adjusted = angle_changes[:len(all_data)]
elif len(angle_changes) < len(all_data):
    print("angle_changes 的数据点少于 all_data 的行数，请检查数据来源。")
else:
    angle_changes_adjusted = angle_changes


all_data['angle_change'] = angle_changes_adjusted


In [ ]:
print(all_data[['kpt_31', 'kpt_32', 'kpt_34', 'kpt_35', 'kpt_52','kpt_53', 'angle_change']])


In [ ]:
average_angle_change = sum(angle_changes) / len(angle_changes)


In [ ]:
print(f"平均角度: {average_angle_change}")


In [ ]:
all_data.to_csv('角度.csv', index=False)
print(f"角度變化計算完成，結果已保存到 'kpts_angle_change.csv'")
files.download('角度.csv')
